In [0]:
%sql
SELECT *
FROM bronze.medicamento
LIMIT 20;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre,
    tipo,
    fabricante,
    stock
FROM bronze.medicamento;

In [0]:
import requests

url = "https://api.fda.gov/drug/label.json"

params = {
    "search": 'openfda.generic_name:"omeprazole"',
    "limit": 3
}

respuesta = requests.get(url, params=params)

print("Código:", respuesta.status_code)
print(respuesta.text[:3000])

In [0]:
%sql
SELECT
    id_medicamento,
    nombre,
    tipo,
    fabricante,
    stock
FROM bronze.medicamento
ORDER BY id_medicamento;

In [0]:
import requests
import json
from pyspark.sql import Row

In [0]:
homologacion = {
    1: "acetaminophen",
    2: "ibuprofen",
    3: "amoxicillin",
    4: "omeprazole",
    5: "losartan",
    6: "metformin",
    7: "albuterol",
    8: "atorvastatin",
    9: "amlodipine",
    10: "azithromycin",
    11: "loratadine",
    12: "diclofenac",
    13: "enalapril",
    14: "acetaminophen",
    15: "omeprazole",
    16: "ceftriaxone",
    17: "insulin human",
    18: "prednisone",
    19: "budesonide",
    20: "amoxicillin and clavulanate potassium"
}

In [0]:
medicamentos = spark.sql("""
    SELECT
        id_medicamento,
        nombre,
        tipo,
        fabricante,
        stock
    FROM bronze.medicamento
    ORDER BY id_medicamento
""")

medicamentos.show(20, False)

In [0]:
url = "https://api.fda.gov/drug/label.json"

resultados = []

for medicamento in medicamentos.collect():

    id_medicamento = medicamento["id_medicamento"]
    nombre = medicamento["nombre"]

    nombre_api = homologacion.get(id_medicamento)

    params = {
        "search": 'openfda.generic_name:"' + nombre_api + '"',
        "limit": 1
    }

    respuesta = requests.get(url, params=params)

    if respuesta.status_code == 200:

        datos = respuesta.json()

        if "results" in datos and len(datos["results"]) > 0:

            resultado_api = datos["results"][0]

            resultados.append(
                Row(
                    id_medicamento=id_medicamento,
                    nombre_medicamento=nombre,
                    nombre_api=nombre_api,
                    codigo_respuesta=respuesta.status_code,
                    encontrado=True,
                    raw_json=json.dumps(resultado_api)
                )
            )

        else:

            resultados.append(
                Row(
                    id_medicamento=id_medicamento,
                    nombre_medicamento=nombre,
                    nombre_api=nombre_api,
                    codigo_respuesta=respuesta.status_code,
                    encontrado=False,
                    raw_json=None
                )
            )

    else:

        resultados.append(
            Row(
                id_medicamento=id_medicamento,
                nombre_medicamento=nombre,
                nombre_api=nombre_api,
                codigo_respuesta=respuesta.status_code,
                encontrado=False,
                raw_json=None
            )
        )

print("Medicamentos procesados:", len(resultados))

In [0]:
df_openfda = spark.createDataFrame(resultados)

df_openfda.write \
    .mode("overwrite") \
    .saveAsTable("bronze.openfda_medicamentos")

In [0]:
%sql
SELECT *
FROM bronze.openfda_medicamentos;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre_medicamento,
    nombre_api,
    codigo_respuesta,
    encontrado
FROM bronze.openfda_medicamentos
ORDER BY id_medicamento;

In [0]:
%sql
SELECT
    encontrado,
    COUNT(*) AS cantidad
FROM bronze.openfda_medicamentos
GROUP BY encontrado;

In [0]:
%sql
SELECT COUNT(*) AS total
FROM bronze.openfda_medicamentos;

In [0]:
%sql
SELECT
    id_medicamento,
    nombre_medicamento,
    nombre_api,
    raw_json
FROM bronze.openfda_medicamentos
WHERE encontrado = true
LIMIT 2;

In [0]:
%sql
DESCRIBE bronze.receta;

In [0]:
%sql
DESCRIBE bronze.detalle_receta;

In [0]:
%sql
SELECT *
FROM bronze.receta
LIMIT 5;

In [0]:
%sql
SELECT *
FROM bronze.detalle_receta
LIMIT 5;

In [0]:
print ("Actualizado")

In [0]:
import requests
import pandas as pd

In [0]:
client_id = "32c7c731-9bcf-4c6b-9ec4-f4cb165ac2a3_78c38d5c-faf9-4875-82ba-a112b035c384"
client_secret = "lVXLZEY/Chh1BDv5zEVpiT1yaSau4pRNcH8Kz/0bmLs="

token_url = "https://icdaccessmanagement.who.int/connect/token"

data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "icdapi_access"
}

response = requests.post(token_url, data=data)

print("Código:", response.status_code)
print(response.text[:1000])

In [0]:
import requests

token_url = "https://icdaccessmanagement.who.int/connect/token"

data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "icdapi_access"
}

response = requests.post(
    token_url,
    data=data
)

print("Código:", response.status_code)

In [0]:
access_token = response.json()["access_token"]

headers = {
    "Authorization": f"Bearer {access_token}",
    "API-Version": "v2",
    "Accept": "application/json"
}

print("Token y headers listos")

In [0]:
headers = {
    "Authorization": f"Bearer {access_token}",
    "API-Version": "v2",
    "Accept": "application/json",
    "Accept-Language": "es"
}

url = "https://id.who.int/icd/release/11/2026-01/mms"

icd_response = requests.get(
    url,
    headers=headers
)

print("Código:", icd_response.status_code)
print(icd_response.text[:3000])

In [0]:
diagnostico = "Hipertensión arterial"

url = "https://id.who.int/icd/release/11/2026-01/mms/search"

params = {
    "q": diagnostico
}

who_response = requests.get(
    url,
    headers=headers,
    params=params
)

print("Código:", who_response.status_code)
print(who_response.text[:3000])

In [0]:
headers = {
    "Authorization": f"Bearer {access_token}",
    "API-Version": "v2",
    "Accept": "application/json",
    "Accept-Language": "es"
}

print("Headers listos")

In [0]:
import requests
import pandas as pd
import time
import re
import html

In [0]:
diagnosticos_cmi = spark.sql("""
    SELECT
        id_diagnostico,
        nombre,
        descripcion
    FROM bronze.diagnostico
    WHERE nombre IS NOT NULL
""")

display(diagnosticos_cmi)

In [0]:
diagnostico = "Hipertensión arterial"

url_prueba = "https://id.who.int/icd/release/11/2026-01/mms/search"

params_prueba = {
    "q": diagnostico
}

respuesta_prueba = requests.get(
    url_prueba,
    headers=headers,
    params=params_prueba
)

print("URL utilizada:")
print(respuesta_prueba.url)

print("\nCódigo:")
print(respuesta_prueba.status_code)

print("\nRespuesta:")
print(respuesta_prueba.text[:3000])

In [0]:
import requests
import pandas as pd
import time
import re
import html

# Obtener los diagnósticos de Centro Médico Integral
diagnosticos_cmi = spark.sql("""
    SELECT 
        id_diagnostico,
        nombre,
        descripcion
    FROM bronze.diagnostico
    WHERE nombre IS NOT NULL
""")

display(diagnosticos_cmi)

In [0]:
resultados_who = []

url_who = "https://id.who.int/icd/release/11/2026-01/mms/search"

for fila in diagnosticos_cmi.collect():

    id_diagnostico = fila["id_diagnostico"]
    nombre = fila["nombre"]
    descripcion = fila["descripcion"]

    print("Consultando:", nombre)

    params = {
        "q": nombre
    }

    response = requests.get(
        url_who,
        headers=headers,
        params=params
    )

    if response.status_code == 200:

        data = response.json()

        resultados = data.get("destinationEntities", [])

        if len(resultados) > 0:

            # Buscar el resultado con mayor score
            mejor = max(
                resultados,
                key=lambda x: x.get("score", 0)
            )

            titulo = mejor.get("title")

            if titulo:
                titulo = html.unescape(titulo)
                titulo = re.sub(r"<[^>]+>", "", titulo)

            resultados_who.append({
                "id_diagnostico": id_diagnostico,
                "diagnostico_cmi": nombre,
                "descripcion_cmi": descripcion,
                "nombre_who": titulo,
                "codigo_icd11": mejor.get("theCode"),
                "id_who": mejor.get("id"),
                "score": mejor.get("score")
            })

            print(
                "  ✓",
                titulo,
                "| Código:",
                mejor.get("theCode"),
                "| Score:",
                mejor.get("score")
            )

        else:

            resultados_who.append({
                "id_diagnostico": id_diagnostico,
                "diagnostico_cmi": nombre,
                "descripcion_cmi": descripcion,
                "nombre_who": None,
                "codigo_icd11": None,
                "id_who": None,
                "score": None
            })

            print("  ⚠ No se encontró coincidencia")

    else:

        print("  ✗ Error:", response.status_code)

        resultados_who.append({
            "id_diagnostico": id_diagnostico,
            "diagnostico_cmi": nombre,
            "descripcion_cmi": descripcion,
            "nombre_who": None,
            "codigo_icd11": None,
            "id_who": None,
            "score": None
        })

    time.sleep(0.5)

In [0]:
df_who = pd.DataFrame(resultados_who)

display(df_who)

In [0]:
spark_who = spark.createDataFrame(df_who)

spark_who.write \
    .mode("overwrite") \
    .saveAsTable("bronze.who_icd11_diagnosticos")

print("✅ Tabla creada: bronze.who_icd11_diagnosticos")

In [0]:
%sql

SELECT *
FROM bronze.who_icd11_diagnosticos
ORDER BY id_diagnostico;

In [0]:
import requests

url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"

params = {
    "name": "omeprazole"
}

response = requests.get(url, params=params)

print("Código:", response.status_code)
print(response.text[:3000])

In [0]:
medicamentos_cmi = spark.sql("""
    SELECT DISTINCT nombre
    FROM bronze.medicamento
    WHERE nombre IS NOT NULL
""")

display(medicamentos_cmi)

In [0]:
import re
import unicodedata

def limpiar_nombre(nombre):
    nombre = unicodedata.normalize("NFD", nombre)
    nombre = "".join(
        c for c in nombre
        if unicodedata.category(c) != "Mn"
    )
    
    nombre = nombre.lower()
    
    nombre = re.sub(
        r'\d+\s*(mg|mcg|g|ml|%|ui)\b',
        '',
        nombre
    )
    
    nombre = re.sub(r'\s+', ' ', nombre).strip()
    
    return nombre

In [0]:
def buscar_rxnorm(nombre):

    nombre_limpio = limpiar_nombre(nombre)

    url = "https://rxnav.nlm.nih.gov/REST/rxcui.json"

    params = {
        "name": nombre_limpio
    }

    response = requests.get(
        url,
        params=params
    )

    if response.status_code == 200:

        data = response.json()

        ids = data.get(
            "idGroup",
            {}
        ).get(
            "rxnormId",
            []
        )

        if len(ids) > 0:
            return ids[0]

    return None

In [0]:
resultados_rxnorm = []

for fila in medicamentos_cmi.collect():

    medicamento = fila["nombre"]

    print("Consultando:", medicamento)

    rxcui = buscar_rxnorm(medicamento)

    resultados_rxnorm.append({
        "medicamento_cmi": medicamento,
        "rxcui": rxcui
    })

In [0]:
df_rxnorm = pd.DataFrame(resultados_rxnorm)

display(df_rxnorm)

In [0]:
def obtener_nombre_rxnorm(rxcui):

    if rxcui is None:
        return None

    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}.json"

    response = requests.get(url)

    if response.status_code == 200:

        data = response.json()

        return data.get(
            "idGroup",
            {}
        ).get("name")

    return None

In [0]:
df_rxnorm["nombre_normalizado"] = df_rxnorm["rxcui"].apply(
    obtener_nombre_rxnorm
)

display(df_rxnorm)

In [0]:
spark_rxnorm = spark.createDataFrame(df_rxnorm)

spark_rxnorm.write \
    .mode("overwrite") \
    .saveAsTable("bronze.rxnorm_medicamentos")

print("✅ Tabla creada: bronze.rxnorm_medicamentos")

In [0]:
%sql

SELECT *
FROM bronze.rxnorm_medicamentos;

In [0]:
import requests
from urllib.parse import quote

nombre = "omeprazole"

url = (
    "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
    "compound/name/"
    + quote(nombre, safe="")
    + "/property/MolecularFormula,MolecularWeight,InChIKey/JSON"
)

response = requests.get(url)

print("Código:", response.status_code)
print(response.text[:3000])

In [0]:
def buscar_pubchem(nombre):

    nombre_limpio = limpiar_nombre(nombre)

    url = (
        "https://pubchem.ncbi.nlm.nih.gov/rest/pug/"
        "compound/name/"
        + quote(nombre_limpio, safe="")
        + "/property/"
        "MolecularFormula,MolecularWeight,InChIKey/JSON"
    )

    response = requests.get(url)

    if response.status_code == 200:

        data = response.json()

        propiedades = (
            data
            .get("PropertyTable", {})
            .get("Properties", [])
        )

        if len(propiedades) > 0:
            return propiedades[0]

    return None

In [0]:
resultados_pubchem = []

for fila in medicamentos_cmi.collect():

    medicamento = fila["nombre"]

    print("Consultando:", medicamento)

    resultado = buscar_pubchem(medicamento)

    if resultado:

        resultados_pubchem.append({
            "medicamento_cmi": medicamento,
            "cid": resultado.get("CID"),
            "formula_molecular": resultado.get(
                "MolecularFormula"
            ),
            "peso_molecular": resultado.get(
                "MolecularWeight"
            ),
            "inchikey": resultado.get(
                "InChIKey"
            )
        })

    else:

        resultados_pubchem.append({
            "medicamento_cmi": medicamento,
            "cid": None,
            "formula_molecular": None,
            "peso_molecular": None,
            "inchikey": None
        })

In [0]:
df_pubchem = pd.DataFrame(resultados_pubchem)

display(df_pubchem)

In [0]:
spark_pubchem = spark.createDataFrame(df_pubchem)

spark_pubchem.write \
    .mode("overwrite") \
    .saveAsTable("bronze.pubchem_medicamentos")

print("✅ Tabla creada: bronze.pubchem_medicamentos")

In [0]:
%sql

SELECT *
FROM bronze.pubchem_medicamentos;

In [0]:
import requests

url = "https://clinicaltrials.gov/api/v2/studies"

params = {
    "query.intr": "omeprazole",
    "pageSize": 5,
    "format": "json"
}

response = requests.get(
    url,
    params=params
)

print("Código:", response.status_code)
print(response.text[:3000])

In [0]:
def buscar_clinical_trials(nombre):

    url = "https://clinicaltrials.gov/api/v2/studies"

    params = {
        "query.intr": limpiar_nombre(nombre),
        "pageSize": 20,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params
    )

    if response.status_code != 200:
        return []

    data = response.json()

    estudios = []

    for estudio in data.get("studies", []):

        protocolo = estudio.get(
            "protocolSection",
            {}
        )

        identificacion = protocolo.get(
            "identificationModule",
            {}
        )

        estado = protocolo.get(
            "statusModule",
            {}
        )

        diseño = protocolo.get(
            "designModule",
            {}
        )

        intervenciones = protocolo.get(
            "armsInterventionsModule",
            {}
        )

        condiciones = protocolo.get(
            "conditionsModule",
            {}
        )

        estudios.append({

            "medicamento_cmi": nombre,

            "nct_id": identificacion.get(
                "nctId"
            ),

            "titulo": identificacion.get(
                "briefTitle"
            ),

            "estado": estado.get(
                "overallStatus"
            ),

            "tipo_estudio": diseño.get(
                "studyType"
            ),

            "fase": str(
                diseño.get("phases")
            ),

            "condiciones": str(
                condiciones.get("conditions")
            ),

            "intervenciones": str(
                intervenciones.get("interventions")
            )
        })

    return estudios

In [0]:
resultados_trials = []

for fila in medicamentos_cmi.collect():

    medicamento = fila["nombre"]

    print("Consultando:", medicamento)

    estudios = buscar_clinical_trials(
        medicamento
    )

    resultados_trials.extend(
        estudios
    )

In [0]:
df_trials = pd.DataFrame(
    resultados_trials
)

display(df_trials)

In [0]:
spark_trials = spark.createDataFrame(
    df_trials
)

spark_trials.write \
    .mode("overwrite") \
    .saveAsTable(
        "bronze.clinicaltrials_medicamentos"
    )

print(
    "✅ Tabla creada: "
    "bronze.clinicaltrials_medicamentos"
)

In [0]:
%sql

SELECT *
FROM bronze.clinicaltrials_medicamentos;